In [1]:
from pathlib import Path
import pandas as pd

# =========================
# 路徑：請改成你的實際路徑
# =========================
base_dir = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Decomposition")

bame_csv = base_dir / "bame_decomp_tidy.csv"
bip_csv  = base_dir / "bip_decomp_tidy.csv"

bame_tex = base_dir / "bame_decomp_table.tex"
bip_tex  = base_dir / "bip_decomp_table.tex"


def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def fmt_coef(x, p, row):
    if pd.isna(x):
        return ""
    if row == "Observations":
        return f"{int(round(x)):,}"
    return f"{x:.3f}{stars(p)}"


def fmt_se(se, row):
    if pd.isna(se) or row == "Observations":
        return ""
    return f"({se:.3f})"


def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["panel"] = df["panel"].astype(str).str.strip()
    df["row"]   = df["row"].astype(str).str.strip()
    df["col"]   = pd.to_numeric(df["col"], errors="coerce").astype("Int64")
    df["coef"]  = pd.to_numeric(df["coef"], errors="coerce")
    df["se"]    = pd.to_numeric(df["se"], errors="coerce")
    df["p"]     = pd.to_numeric(df["p"], errors="coerce")
    return df


def build_cell_dict(df: pd.DataFrame):
    cell = {}
    for _, r in df.iterrows():
        key = (r["panel"], r["row"], int(r["col"]))
        cell[key] = {
            "coef": fmt_coef(r["coef"], r["p"], r["row"]),
            "se": fmt_se(r["se"], r["row"]),
        }
    return cell


def get_coef(cell, panel, row, col):
    return cell.get((panel, row, col), {}).get("coef", "")


def get_se(cell, panel, row, col):
    return cell.get((panel, row, col), {}).get("se", "")


def build_table(
    input_csv: Path,
    output_tex: Path,
    caption: str,
    label: str,
    gap_note: str,
    ref_price_note: str,
):
    df = pd.read_csv(input_csv)
    df = clean_df(df)

    expected_rows = [
        ("overall", "Group gap"),

        ("explained", "Employment and financial"),
        ("explained", "Housing"),
        ("explained", "Health"),
        ("explained", "Household composition"),
        ("explained", "Sex"),
        ("explained", "Age"),
        ("explained", "Total composition effect"),

        ("unexplained", "Employment and financial"),
        ("unexplained", "Housing"),
        ("unexplained", "Health"),
        ("unexplained", "Household composition"),
        ("unexplained", "Sex"),
        ("unexplained", "Age"),
        ("unexplained", "Constant"),
        ("unexplained", "Total structural effect"),

        ("footer", "Observations"),
    ]

    label_map = {
        ("overall", "Group gap"): "Group gap",

        ("explained", "Employment and financial"): r"\hspace{0.1cm}Employment and financial",
        ("explained", "Housing"): r"\hspace{0.1cm}Housing",
        ("explained", "Health"): r"\hspace{0.1cm}Health",
        ("explained", "Household composition"): r"\hspace{0.1cm}Household composition",
        ("explained", "Sex"): r"\hspace{0.1cm}Sex",
        ("explained", "Age"): r"\hspace{0.1cm}Age",
        ("explained", "Total composition effect"): "Total composition effect",

        ("unexplained", "Employment and financial"): r"\hspace{0.1cm}Employment and financial",
        ("unexplained", "Housing"): r"\hspace{0.1cm}Housing",
        ("unexplained", "Health"): r"\hspace{0.1cm}Health",
        ("unexplained", "Household composition"): r"\hspace{0.1cm}Household composition",
        ("unexplained", "Sex"): r"\hspace{0.1cm}Sex",
        ("unexplained", "Age"): r"\hspace{0.1cm}Age",
        ("unexplained", "Constant"): r"\hspace{0.1cm}Constant",
        ("unexplained", "Total structural effect"): "Total structural effect",

        ("footer", "Observations"): "Observations",
    }

    cell = build_cell_dict(df)

    lines = []
    lines.append(r"\begin{sidewaystable}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\label{" + label + "}")
    lines.append(r"\begin{threeparttable}")
    lines.append(r"\scriptsize")
    lines.append(r"\setlength{\tabcolsep}{4pt}")
    lines.append(r"\renewcommand{\arraystretch}{1.08}")
    lines.append(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.0cm}ccccc}")
    lines.append(r"\toprule")
    lines.append(r" & (1) & (2) & (3) & (4) & (5) \\")
    lines.append(
        r" & All factors & Ref. price & Housing only & Housing + HH + demo & All except housing \\"
    )
    lines.append(r"\midrule")

    # overall
    panel, row = ("overall", "Group gap")
    lines.append(
        f"{label_map[(panel, row)]} & "
        f"{get_coef(cell, panel, row, 1)} & {get_coef(cell, panel, row, 2)} & "
        f"{get_coef(cell, panel, row, 3)} & {get_coef(cell, panel, row, 4)} & "
        f"{get_coef(cell, panel, row, 5)} \\\\"
    )
    lines.append(
        f" & {get_se(cell, panel, row, 1)} & {get_se(cell, panel, row, 2)} & "
        f"{get_se(cell, panel, row, 3)} & {get_se(cell, panel, row, 4)} & "
        f"{get_se(cell, panel, row, 5)} \\\\"
    )

    # explained
    lines.append(r"\addlinespace[0.3em]")
    lines.append(r"\multicolumn{6}{l}{\textit{Composition effect attributable to:}} \\")
    for panel, row in expected_rows[1:8]:
        lines.append(
            f"{label_map[(panel, row)]} & "
            f"{get_coef(cell, panel, row, 1)} & {get_coef(cell, panel, row, 2)} & "
            f"{get_coef(cell, panel, row, 3)} & {get_coef(cell, panel, row, 4)} & "
            f"{get_coef(cell, panel, row, 5)} \\\\"
        )
        lines.append(
            f" & {get_se(cell, panel, row, 1)} & {get_se(cell, panel, row, 2)} & "
            f"{get_se(cell, panel, row, 3)} & {get_se(cell, panel, row, 4)} & "
            f"{get_se(cell, panel, row, 5)} \\\\"
        )

    # unexplained
    lines.append(r"\addlinespace[0.3em]")
    lines.append(r"\multicolumn{6}{l}{\textit{Structural effect attributable to:}} \\")
    for panel, row in expected_rows[8:16]:
        lines.append(
            f"{label_map[(panel, row)]} & "
            f"{get_coef(cell, panel, row, 1)} & {get_coef(cell, panel, row, 2)} & "
            f"{get_coef(cell, panel, row, 3)} & {get_coef(cell, panel, row, 4)} & "
            f"{get_coef(cell, panel, row, 5)} \\\\"
        )
        lines.append(
            f" & {get_se(cell, panel, row, 1)} & {get_se(cell, panel, row, 2)} & "
            f"{get_se(cell, panel, row, 3)} & {get_se(cell, panel, row, 4)} & "
            f"{get_se(cell, panel, row, 5)} \\\\"
        )

    # footer
    panel, row = ("footer", "Observations")
    lines.append(r"\addlinespace[0.3em]")
    lines.append(
        f"{label_map[(panel, row)]} & "
        f"{get_coef(cell, panel, row, 1)} & {get_coef(cell, panel, row, 2)} & "
        f"{get_coef(cell, panel, row, 3)} & {get_coef(cell, panel, row, 4)} & "
        f"{get_coef(cell, panel, row, 5)} \\\\"
    )

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular*}")
    lines.append(r"\begin{tablenotes}[flushleft]")
    lines.append(r"\scriptsize")
    lines.append(
        r"\item Notes: Dependent variable is the individual change in standardized, seasonally adjusted and inverted GHQ Likert score. "
        r"Standard errors clustered at the primary sampling unit are reported in parentheses. "
        + gap_note + " "
        + ref_price_note + " "
        + r"Columns (1), (3), (4), and (5) use pooled-price Oaxaca--Blinder decompositions. "
        + r"Column (2) uses the reference-group price. "
        + r"The lower panel groups detailed structural effects for ease of exposition. "
        + r"Employment no-change includes respondents not working before the pandemic. "
        + r"Age 70+ is the omitted category. "
        + r"* \(p<0.10\), ** \(p<0.05\), *** \(p<0.01\)."
    )
    lines.append(r"\end{tablenotes}")
    lines.append(r"\end{threeparttable}")
    lines.append(r"\end{sidewaystable}")

    output_tex.write_text("\n".join(lines), encoding="utf-8")
    print(f"Saved: {output_tex}")


# =========
# BAME 表
# =========
build_table(
    input_csv=bame_csv,
    output_tex=bame_tex,
    caption="Decomposition of the Non-BAME--BAME Gap in Mental Well-Being",
    label="tab:bame_decomp",
    gap_note="Reported gaps are defined as Non-BAME minus BAME.",
    ref_price_note="In Column (2), the decomposition is evaluated at Non-BAME prices.",
)

# ========
# BIP 表
# ========
build_table(
    input_csv=bip_csv,
    output_tex=bip_tex,
    caption="Decomposition of the Non-BIP--BIP Gap in Mental Well-Being",
    label="tab:bip_decomp",
    gap_note="Reported gaps are defined as Non-BIP minus BIP.",
    ref_price_note="In Column (2), the decomposition is evaluated at Non-BIP prices.",
)

Saved: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Decomposition/bame_decomp_table.tex
Saved: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Decomposition/bip_decomp_table.tex
